In [72]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='xgboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始

display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化XGBoost回归器
model = XGBRegressor(random_state=200)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(1,7,1)],#1,3,5
            'n_estimators':[50*n for n in range(1,5)],#50,100,150,200
            'learning_rate':[0.01,0.1,0.2], #0.01,0.1,0.2
            'gamma':[0.1,0.2]} #

# 可修改
model = XGBRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_
best_parameter

# 可修改
model = XGBRegressor(gamma=best_parameter.gamma,learning_rate=best_parameter.learning_rate,max_depth=best_parameter.max_depth)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)

    

文件夹models/xgboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
5,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
754,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
755,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
756,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
757,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


xgboost_tree_feature_names(top15):
MagpieData mean GSvolume_pa
Interant electrons
Ag fraction
Ca fraction
MagpieData range GSvolume_pa
MagpieData maximum NValence
Zr fraction
MagpieData mean NfValence
MagpieData mean AtomicWeight
Zn fraction
MagpieData avg_dev NUnfilled
Interant d electrons
MagpieData range MendeleevNumber
MagpieData maximum CovalentRadius
Yb fraction
average_xgboost_feature_values(top15):
0.13371329
0.13241756
0.06118004
0.040560994
0.038347613
0.0374437
0.03233641
0.026256723
0.021671213
0.02135956
0.020290153
0.019139731
0.01905379
0.01850672
0.017635906
平均 R2 分数: 0.6569001555238101
平均 MAPE 分数: 13.384622378167915
平均MAE分数: 33.55348385154976


In [64]:
print(X.shape)
print(best_parameter)

(739, 277)
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.1, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=50, n_jobs=None,
             num_parallel_tree=None, random_state=200, ...)


In [73]:
# 看看随便更新下的参数结果
model = XGBRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)

xgboost_tree_feature_names(top15):-------+
Interant electrons
MagpieData mean GSvolume_pa
MagpieData range GSvolume_pa
Ca fraction
MagpieData maximum NValence
Zr fraction
Yb fraction
Electronegativity local mismatch
Y fraction
MagpieData mean NfValence
Interant d electrons
Zn fraction
MagpieData maximum CovalentRadius
MagpieData range MendeleevNumber
MagpieData minimum MendeleevNumber
average_xgboost_feature_values(top15):
0.2787183
0.08933093
0.062687494
0.05575772
0.051967848
0.04278889
0.037098303
0.02667767
0.025974227
0.024752522
0.019936454
0.017867759
0.017846854
0.014533137
0.012480591
平均 R2 分数: 0.6395501917523635
平均 MAPE 分数: 13.640175612871275
平均MAE分数: 34.266662854080494


In [74]:
joblib.dump(model,'models/xgboost/xgboost_optimization2.pkl')

['models/xgboost/xgboost_optimization2.pkl']

再进一步更新参数

In [66]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='xgboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化XGBoost回归器
model = XGBRegressor(random_state=200)# 设置随机种子以确保结果的可重复性
# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[3],#1,3,5
            'n_estimators':[200],#50,100,150,200
            'learning_rate':np.arange(0.08, 0.16, 0.01), #0.01,0.1,0.2
            'gamma':np.arange(0.08, 0.16, 0.02)} #

# 可修改
model = XGBRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = XGBRegressor(n_estimators=best_parameter.n_estimators,gamma=best_parameter.gamma,learning_rate=best_parameter.learning_rate,max_depth=best_parameter.max_depth)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization3.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/xgboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'gamma': 0.08, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.09, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': None, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}
xgboost_tree_feature_names(top15):
MagpieData mean GSvolume_pa
MagpieData avg_dev NValence
Interant ele

In [ ]:
model.get_params()

RF

In [69]:
from sklearn.model_selection import GridSearchCV

# 这里采用rf算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='rf'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[25*n for n in range(3,8)],# DT个数 50,100,150,200
            'min_samples_leaf':[1], # 叶子结点所需最小样本数
            'min_samples_split':[2],#分裂最小样本数
            'max_features':['sqrt','log2',1],
            'criterion':['gini','squared_error']
            } #最大特征数量

# 可修改
model = RandomForestRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = RandomForestRegressor(random_state=best_parameter.random_state, min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)

    

文件夹models/rf已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 7, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 200, 'verbose': 0, 'warm_start': False}
rf_tree_feature_names(top15):
Calculated Yield Strength
Grain Size
Calculated Grain Boundary
Habit Plane
Interant electrons
frac f valence electrons
Interant d electrons
Gd fraction
MagpieData mean NfUnfilled
MagpieData mean GSvolume_pa
Yang delta
Radii gamma
MagpieData avg_dev NfUnfilled
mean AtomicRadius
MagpieData avg_dev NfValence
average_rf_feature_values(top15):
0.07063423796623274
0.06767916248064995
0.06442292408400156
0.04070300051107239
0.03315212344760755
0.020607005787264564
0.02036289001990186
0.019080098185243498
0.017796316242649265
0.014895611114978774
0.013792728596800358
0.012

In [70]:
from sklearn.model_selection import GridSearchCV    

# 这里采用rf算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='rf'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
model = RandomForestRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

文件夹models/rf已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


rf_tree_feature_names(top15):-------+
Grain Size
Calculated Yield Strength
Habit Plane
Interant electrons
MagpieData mean GSvolume_pa
Calculated Grain Boundary
Interant d electrons
Zr fraction
frac f valence electrons
MagpieData range GSvolume_pa
Lambda entropy
Yang omega
Mixing enthalpy
Ca fraction
Y fraction
average_rf_feature_values(top15):
0.23236491803220477
0.09425049795267283
0.0733117292865112
0.07297997174597198
0.04639457830434458
0.042573602612735705
0.034898221010688596
0.023886009139651193
0.018751069398605137
0.018586226485026974
0.01646070146123867
0.015912161001005645
0.015564061050514144
0.014107454052149112
0.012731811872814157
平均 R2 分数: 0.6684024328330826
平均 MAPE 分数: 13.689584187734749
平均MAE分数: 33.91644536711449


{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 200,
 'verbose': 0,
 'warm_start': False}

In [71]:
joblib.dump(model,'models/rf/rf_best2.pkl')

['models/rf/rf_best2.pkl']

再次得到最优参数，发现效果并不好

In [37]:
# from sklearn.model_selection import GridSearchCV
# 
# # 这里采用rf算法进行预测
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
# from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
# import joblib
# from openpyxl import load_workbook
# import os
# """
# 路径的配置和选择
# """
# model_name='rf'
# 
# save_model=True
# del_0qf=True #是否删除0屈服强度数据
# main_path=f'models/{model_name}' # 这里是模型所在的文件位置
# create_dir(main_path,is_mainpath=True)
# # excel='index/xgboost_index.xlsx'
# # 如果分数表格不存在就新建一个
# score_path='scores_with_best_parameters.xlsx'
# if os.path.exists(score_path):
#     print(f"The file '{score_path}' exists.")
# else:
#     print(f"The file '{score_path}' does not exist.")
#     df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
#     df.to_excel(score_path, index=False)
# 
# 
# """
# 文件读取部分
# """
# # 读取 Excel 文件
# df = pd.read_excel('train_set.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# # 删除包含空值的行
# if del_0qf:
#     df = df[df['屈服强度'] != 0].reset_index(drop=True)
#     # df = df[df['计算晶界'] != 0].reset_index(drop=True)
#     # df = df[df['计算固溶'] != 0].reset_index(drop=True)
# display(df)
# np.random.seed(200) #设置随机数种子
# feature_names = df.columns[:-4]  # 特征名为屈服强度（倒数第四列）之前的部分
# X = df.iloc[:, :-4]  # 特征: 最后四列之前的所有列
# y = df.iloc[:, -4]  # 目标: 倒数第四列
# # 将目标变量分箱为10个类别
# bins = np.linspace(y.min(), y.max(), 11)
# y_binned = np.digitize(y, bins)
# # print('y_binned',y_binned)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)
# 
# """
# 设置初始化
# """
# n_splits=5
# # 初始化StratifiedKFold
# kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同
# 
# # 初始化列表以存储每个折叠的分数
# # 定义评估指标
# scoring = {
#     'r2':make_scorer(r2_score,greater_is_better=True),
#     'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
#     'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
# }
# r2_scores_ls = []
# mape_scores = []
# mae_scores=[]
# best_r2_score = float('-inf')
# best_mae_score= float('-inf')
# best_mape_score= float('-inf')
# ls1=['fold'+str(i) for i in range(1,n_splits+1)]
# fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))
# 
# # 可修改
# parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
#             'n_estimators':[100,125,150],# DT个数 50,100,150,200
#             'min_samples_leaf':[1], # 叶子结点所需最小样本数
#             'min_samples_split':[2],#分裂最小样本数
#             'max_features':[1],
#             'criterion':['squared_error']
#             } #最大特征数量
# 
# # 可修改
# model = RandomForestRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
# grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
# grid_search.fit(X_train,y_train)
# best_parameter=grid_search.best_estimator_
# 
# 
# # 可修改
# model = RandomForestRegressor(random_state=best_parameter.random_state, min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
# model.fit(X_train, y_train)
# print(model.get_params())
# # 计算分折交叉验证结果
# results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# # 输出分折交叉验证结果
# for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
#     # 记录不同折的分数结果
#     r2_scores_ls.append(r2)
#     mape_scores.append(-mape)
#     mae_scores.append(-mae)
#     # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")
# 
# current_r2_score=np.mean(r2_scores_ls)
# # scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# # print('r2_scores:',scores)
# # current_r2_score=scores.mean()
# # 只对更优的结果进行保存
# # 记录模型的特征重要性
# feature_importance_final=model.feature_importances_
# importance_dict = dict(zip(feature_importance_final, feature_names))
# importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
# importance_sorted = sorted(importance_tuplelist, reverse=True)
# x_feature = [j for (i, j) in importance_sorted]
# y_importance_value = [i for (i, j) in importance_sorted]
# print(f'{model_name}_tree_feature_names(top15):')
# for i in x_feature[:15]:
#     print(i)
# print(f'average_{model_name}_feature_values(top15):')
# for i in y_importance_value[:15]:
#     print(i)
# 
# #记录更优的分数
# best_r2_score=current_r2_score
# best_mae_score=np.mean(mae_scores)
# best_mape_score=np.mean(mape_scores)
# if save_model:
#     joblib.dump(model,   f'models/{model_name}/{model_name}_optimization.pkl')  #保存模型
#     # print(f'保存更优{model_name}模型文件') 
# print('平均 R2 分数:', best_r2_score)
# print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
# print('平均MAE分数:',best_mae_score)
# 
# 
#     
# """
# 打印并输出分数至表格
# """
# 
# df = pd.read_excel(score_path)
# # 添加新数据的示例
# new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
# new_data_df = pd.DataFrame([new_data])
# 
# # 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
# df = pd.concat([df, new_data_df], ignore_index=True)
# # 保存数据框到Excel表格
# df.to_excel(score_path, index = False)

# 
#     

GBoost

In [38]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='gboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)

"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       'Habit Plane','屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
            'min_samples_leaf':[1,2], # 叶子结点所需最小样本数
            'min_samples_split':[2,3,4],#分裂最小样本数
            'max_features':['auto','sqrt','log2','None'],
            'learning_rate':[0.01,0.1,0.2], 
            } #最大特征数量

# 可修改
model = GradientBoostingRegressor(random_state=200) # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = GradientBoostingRegressor(n_estimators=best_parameter.n_estimators,min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,learning_rate=best_parameter.learning_rate)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/gboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'squared_error', 'max_depth': 3, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 2, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 150, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
gboost_tree_feature_names(top15):
Grain Size
Calculated Yield Strength
Calculated Solid Solution
Interant d electrons
MagpieData avg_dev NfUnfilled
mean AtomicRadius
MagpieData avg_dev GSmagmom
mean Electronegativity
MagpieData avg_dev MeltingT
frac f valence electrons
MagpieData mean NdValence
Configuration entropy
Atomic weight mean
Mixing enthalpy
Radii gamma
average_gboost_feature_values(top15):
0.09468782968369852
0.0767450209425087
0.055698368587932236
0.033385659692470114
0.028482799530993454
0.028013823961098194
0.0270809861

In [39]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='gboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       'Habit Plane','屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

model = GradientBoostingRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

文件夹models/gboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


gboost_tree_feature_names(top15):-------+
Grain Size
Calculated Solid Solution
Interant electrons
MagpieData mean GSvolume_pa
Zr fraction
Calculated Yield Strength
Yang delta
Interant d electrons
Y fraction
MagpieData range GSvolume_pa
Radii gamma
frac f valence electrons
Ca fraction
Mixing enthalpy
avg f valence electrons
average_gboost_feature_values(top15):
0.2219728953557341
0.10013425900171823
0.09310592603161053
0.06586847466052222
0.0471071068214554
0.04506788310302906
0.03363111351468702
0.024169013833131504
0.02050160346987083
0.020441831195722914
0.018908195898706244
0.014504583172167579
0.014318456237508333
0.013247302430156909
0.012357685690725883
平均 R2 分数: 0.6647234819474384
平均 MAPE 分数: 13.967965228099372
平均MAE分数: 34.24428800210343


{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 200,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [40]:
joblib.dump(model,'models/gboost/gboost_best1.pkl')

['models/gboost/gboost_best1.pkl']

Gboost参数再优化

In [41]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='gboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
       'Habit Plane','屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,6,1)], # 最大深度5,7,9
            'n_estimators':[25*n for n in range(3,5)],# DT个数 50,100,150,200
            'min_samples_leaf':[1], # 叶子结点所需最小样本数
            'min_samples_split':[2],#分裂最小样本数
            'max_features':['auto','sqrt','log2','None'],
            'learning_rate':np.arange(0.08,0.15,0.01), 
            } #最大特征数量

# 可修改
model = GradientBoostingRegressor(random_state=200) # 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = GradientBoostingRegressor(n_estimators=best_parameter.n_estimators,min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,learning_rate=best_parameter.learning_rate)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization3.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/gboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'alpha': 0.9, 'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.10999999999999999, 'loss': 'squared_error', 'max_depth': 3, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 100, 'n_iter_no_change': None, 'random_state': None, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
gboost_tree_feature_names(top15):
Grain Size
Calculated Yield Strength
Calculated Solid Solution
Interant d electrons
mean Electronegativity
mean AtomicRadius
MagpieData avg_dev GSmagmom
MagpieData avg_dev NfUnfilled
Configuration entropy
MagpieData avg_dev MeltingT
MagpieData mean NdValence
frac f valence electrons
Atomic weight mean
Radii gamma
MagpieData mean NfValence
average_gboost_feature_values(top15):
0.09760553426636179
0.08227980010413771
0.053513930105726686
0.0323214877408239
0.031087318490184767
0.029971

DT`

In [42]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='dt'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
     '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            # 'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
            'min_samples_leaf':[1,2,'None'], # 叶子结点所需最小样本数
            'min_samples_split':[2,3,4,'None'],#分裂最小样本数
            'max_features':['auto','sqrt','log2','None'],
            'criterion':['absolute_error', 'squared_error', 'friedman_mse', 'poisson'] 
            } #最大特征数量

# 可修改
model = DecisionTreeRegressor(random_state=200)
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = DecisionTreeRegressor(min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/dt已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 7, 'max_features': 'log2', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 3, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': None, 'splitter': 'best'}
dt_tree_feature_names(top15):
mean Electronegativity
Calculated Grain Boundary
mean Number
MagpieData maximum GSvolume_pa
MagpieData avg_dev CovalentRadius
Radii gamma
avg f valence electrons
mean Column
Shear modulus delta
MagpieData avg_dev GSmagmom
MagpieData avg_dev MeltingT
MagpieData range GSvolume_pa
Yang delta
MagpieData minimum GSvolume_pa
mean AtomicRadius
average_dt_feature_values(top15):
0.21403366146084815
0.14054400231044295
0.0936623929611783
0.08129662023274957
0.051663644179274194
0.050057110908026815
0.03809316374877709
0.035161919770257864
0.029963113413867197
0.02823539324009548
0.02772945527823716
0.0270233255685201
0.02607811339134161
0.022021504646835353
0.016110781907431097
平均 R2

In [43]:
model = DecisionTreeRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

dt_tree_feature_names(top15):-------+
Grain Size
Interant electrons
MagpieData mean GSvolume_pa
Calculated Yield Strength
Habit Plane
Zr fraction
MagpieData range GSvolume_pa
Calculated Grain Boundary
Y fraction
MagpieData minimum MendeleevNumber
frac d valence electrons
Yang omega
Zn fraction
avg f valence electrons
Ca fraction
average_dt_feature_values(top15):
0.24903495134962092
0.12182829516318376
0.10022313234368244
0.07761595294063356
0.054935936213598525
0.04277165965076696
0.041455467477703865
0.028921911017775363
0.02660868821274468
0.021245262171780323
0.020166486534677446
0.019568414443944127
0.018618810974165134
0.016573302305358143
0.01421364231777376
平均 R2 分数: 0.4120221708347276
平均 MAPE 分数: 17.501069908086595
平均MAE分数: 43.83728558753431


{'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 200,
 'splitter': 'best'}

In [44]:
joblib.dump(model,'models/dt/dt_best.pkl')

['models/dt/dt_best.pkl']

In [45]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='dt'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution',
     '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'max_depth':np.arange(3,6,1), # 最大深度5,7,9
            # 'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
            'min_samples_leaf':[1], # 叶子结点所需最小样本数
            'min_samples_split':[2],#分裂最小样本数
            'max_features':['sqrt','log2'],
            'criterion':[ 'squared_error'] 
            } #最大特征数量

# 可修改
model = DecisionTreeRegressor(random_state=200)
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = DecisionTreeRegressor(min_samples_leaf=best_parameter.min_samples_leaf,min_samples_split=best_parameter.min_samples_split,max_depth=best_parameter.max_depth,max_features=best_parameter.max_features,criterion=best_parameter.criterion)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization3.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


    

文件夹models/dt已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 4, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'random_state': None, 'splitter': 'best'}
dt_tree_feature_names(top15):
mean Electronegativity
Habit Plane
avg d valence electrons
MagpieData avg_dev GSmagmom
Interant f electrons
mean Row
MagpieData avg_dev MeltingT
Shear modulus mean
MagpieData avg_dev NdUnfilled
Mg fraction
MagpieData mean NdValence
MagpieData avg_dev NValence
Calculated Yield Strength
average_dt_feature_values(top15):
0.2488301378905471
0.1266679722932828
0.12238734243490106
0.11330096686255478
0.09878548577286013
0.0840698523980437
0.05270422744565082
0.04014106585461211
0.03384267335596543
0.033323848487338825
0.030564201193273223
0.015382226010970022
0.0
平均 R2 分数: 0.31040642123872725
平均 MAPE 分数: 21.018850950082125
平均MAE分数: 51.689150925217724


AdaBoost

In [46]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import AdaBoostRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='adaboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
feature_names = df.drop(columns=['Precipitate Distribution'
       ,'屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={#'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[50*n for n in range(1,5)],# DT个数 50,100,150,200
            # 'min_samples_leaf':[1,2,'None'], # 叶子结点所需最小样本数
            # 'min_samples_split':[2,3,4,'None'],#分裂最小样本数
            'learning_rate':[0.01,0.05,0.1,0.15,0.2], 
            # 'max_features':['auto','sqrt','log2','None'],
            'loss':['square', 'linear', 'exponential'] 
            } #最大特征数量

# 可修改
model = AdaBoostRegressor(random_state=200)
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = AdaBoostRegressor(n_estimators=best_parameter.n_estimators,learning_rate=best_parameter.learning_rate,loss=best_parameter.loss)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


文件夹models/adaboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'estimator': None, 'learning_rate': 0.05, 'loss': 'exponential', 'n_estimators': 150, 'random_state': None}
adaboost_tree_feature_names(top15):
Grain Size
Interant electrons
Calculated Yield Strength
Habit Plane
MagpieData mean GSvolume_pa
Zr fraction
Calculated Grain Boundary
Interant d electrons
Mixing enthalpy
frac f valence electrons
Y fraction
Lambda entropy
Radii gamma
Ca fraction
MagpieData range GSvolume_pa
average_adaboost_feature_values(top15):
0.20603324720361857
0.16662818934620116
0.13138369941411485
0.0885277759468064
0.05328028911350792
0.03815946443469674
0.03381976167884649
0.029738379311304047
0.02109101762746814
0.019459258465884078
0.014807981804857097
0.011778353184810476
0.011688495226409868
0.011632756111275656
0.011506489024193869
平均 R2 分数: 0.5404807596770516
平均 MAPE 分数: 17.85175918418979
平均MAE分数: 42.43813118790147


In [47]:
best_parameter

AdaBoostRegressor(learning_rate=0.05, loss='exponential', n_estimators=150,
                  random_state=200)

In [48]:
model = AdaBoostRegressor(random_state=200)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):-------+')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)
#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)

print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
model.get_params()

adaboost_tree_feature_names(top15):-------+
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Interant electrons
Habit Plane
Interant d electrons
mean AtomicRadius
Y fraction
Zr fraction
MagpieData mean GSvolume_pa
frac f valence electrons
Lambda entropy
Mixing enthalpy
Radii gamma
MagpieData mean MendeleevNumber
average_adaboost_feature_values(top15):
0.15236408660132975
0.12846862357032254
0.12573319630258678
0.09621841178018317
0.06581260653919062
0.05194129661289231
0.04826179727348915
0.03425713665272196
0.029960893348644328
0.02701405061021354
0.026997435262444077
0.01413031074984585
0.013684154942924413
0.013584335842972086
0.0134416431215961
平均 R2 分数: 0.5273651374362827
平均 MAPE 分数: 18.27821915235834
平均MAE分数: 43.76319428340513


{'estimator': None,
 'learning_rate': 1.0,
 'loss': 'linear',
 'n_estimators': 50,
 'random_state': 200}

Adaboost再调参

In [49]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import AdaBoostRegressor
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='adaboost'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)
    # df = df[df['计算晶界'] != 0].reset_index(drop=True)
    # df = df[df['计算固溶'] != 0].reset_index(drop=True)
display(df)
np.random.seed(200) #设置随机数种子
df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={#'max_depth':[n for n in range(3,8,2)], # 最大深度5,7,9
            'n_estimators':[25*n for n in range(2,4)],# DT个数 50,100,150,200
            # 'min_samples_leaf':[1,2,'None'], # 叶子结点所需最小样本数
            # 'min_samples_split':[2,3,4,'None'],#分裂最小样本数
            'learning_rate':np.arange(0.13,0.2,0.02), 
            # 'max_features':['auto','sqrt','log2','None'],
            'loss':['exponential'] 
            } #最大特征数量

# 可修改
model = AdaBoostRegressor(random_state=200)
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = AdaBoostRegressor(n_estimators=best_parameter.n_estimators,learning_rate=best_parameter.learning_rate,loss=best_parameter.loss)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
print(model.get_params())
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
feature_importance_final=model.feature_importances_
importance_dict = dict(zip(feature_importance_final, feature_names))
importance_tuplelist = [(abs(value), feature) for (value, feature) in importance_dict.items()]
importance_sorted = sorted(importance_tuplelist, reverse=True)
x_feature = [j for (i, j) in importance_sorted]
y_importance_value = [i for (i, j) in importance_sorted]
print(f'{model_name}_tree_feature_names(top15):')
for i in x_feature[:15]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in y_importance_value[:15]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization3.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)


    
"""
打印并输出分数至表格
"""

df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)

    

文件夹models/adaboost已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'estimator': None, 'learning_rate': 0.13, 'loss': 'exponential', 'n_estimators': 75, 'random_state': None}
adaboost_tree_feature_names(top15):
Grain Size
Interant electrons
Calculated Yield Strength
Habit Plane
MagpieData mean GSvolume_pa
Calculated Grain Boundary
Interant d electrons
Zr fraction
Mixing enthalpy
frac f valence electrons
Lambda entropy
Y fraction
Radii gamma
mean AtomicRadius
MagpieData range GSvolume_pa
average_adaboost_feature_values(top15):
0.1865251749631818
0.16882487359652457
0.11076749966904902
0.08501921952572568
0.0722968245862649
0.056023191702223724
0.04053305545189205
0.03574099875425319
0.02659725907085601
0.02385884536029883
0.01293128129194466
0.011399014058702188
0.010596827601624257
0.008869593783317148
0.008608092896046528
平均 R2 分数: 0.535219011128446
平均 MAPE 分数: 17.997145053011643
平均MAE分数: 42.862016238734995


KNN

In [53]:
from sklearn.model_selection import GridSearchCV

# 这里采用xgboost算法进行预测
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold,train_test_split,cross_validate,cross_val_score,KFold
from sklearn.metrics import r2_score, mean_absolute_percentage_error,mean_absolute_error,make_scorer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor  # 导入GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor 
from sklearn.inspection import permutation_importance
from utils import create_empty_ls,create_dir,mycopyfile,mycopydir
import joblib
from openpyxl import load_workbook
import os
"""
路径的配置和选择
"""
model_name='knn'

save_model=True
del_0qf=True #是否删除0屈服强度数据
main_path=f'models/{model_name}' # 这里是模型所在的文件位置
create_dir(main_path,is_mainpath=True)
# excel='index/xgboost_index.xlsx'
# 如果分数表格不存在就新建一个
score_path='scores_with_best_parameters.xlsx'
if os.path.exists(score_path):
    print(f"The file '{score_path}' exists.")
else:
    print(f"The file '{score_path}' does not exist.")
    df = pd.DataFrame(columns=['Model', 'R^2', 'MAPE', 'MAE'])
    df.to_excel(score_path, index=False)


"""
文件读取部分
"""
# 读取 Excel 文件
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始
# 删除包含空值的行
if del_0qf:
    df = df[df['屈服强度'] != 0].reset_index(drop=True)

display(df)
np.random.seed(200) #设置随机数种子
df.drop(columns=['Precipitate Distribution',
       '屈服强度', '抗拉强度 (UTS)', '追踪编号']).columns
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  # 目标: 倒数第四列
# 将目标变量分箱为10个类别
bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
# print('y_binned',y_binned)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=200)

"""
设置初始化
"""
n_splits=5
# 初始化StratifiedKFold
kf = KFold(n_splits=5, shuffle=True, random_state=200)# 注意这里修改了random_state以确保每次初始化不同

# 初始化列表以存储每个折叠的分数
# 定义评估指标
scoring = {
    'r2':make_scorer(r2_score,greater_is_better=True),
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'MAPE': make_scorer(mean_absolute_percentage_error, greater_is_better=False)
}
r2_scores_ls = []
mape_scores = []
mae_scores=[]
best_r2_score = float('-inf')
best_mae_score= float('-inf')
best_mape_score= float('-inf')
ls1=['fold'+str(i) for i in range(1,n_splits+1)]
fold_dict=dict(zip(ls1,[0 for i in range(0,len(ls1))]))

# 可修改
parameters={'n_neighbors':[n for n in range(3,9,1)], # 邻居数量：3,5,7
            'algorithm':['auto','ball_tree','kd_tree','brute'], # 叶子结点所需最小样本数
            'weights':['uniform','distance'],
            'leaf_size':np.arange(10,40,5),#分裂最小样本数
            } #最大特征数量"


# 可修改
model = KNeighborsRegressor() 
# 注意这里修改了random_state以确保每次初始化不同
grid_search=GridSearchCV(model,parameters,scoring='r2',cv=kf) # 选择r2作为性能评估参数
grid_search.fit(X_train,y_train)
best_parameter=grid_search.best_estimator_


# 可修改
model = KNeighborsRegressor(n_neighbors=best_parameter.n_neighbors,algorithm=best_parameter.algorithm,weights=best_parameter.weights,leaf_size=best_parameter.leaf_size)  # 注意这里修改了random_state以确保每次初始化不同
model.fit(X_train, y_train)
# 计算分折交叉验证结果
results = cross_validate(model,X,y,cv=kf, scoring=scoring, return_train_score=False)
print(model.get_params())
# 输出分折交叉验证结果
for i, (r2, mae, mape) in enumerate(zip(results['test_r2'],results['test_MAE'], results['test_MAPE']), start=1):
    # 记录不同折的分数结果
    r2_scores_ls.append(r2)
    mape_scores.append(-mape)
    mae_scores.append(-mae)
    # print(f"折 {i}: r2={r2:.3f},MAE = {-mae:.3f}, MAPE = {-mape:.3f}")

current_r2_score=np.mean(r2_scores_ls)
# scores = cross_val_score(model, X, y_binned, cv=skf, scoring=make_scorer(r2_score))
# print('r2_scores:',scores)
# current_r2_score=scores.mean()
# 只对更优的结果进行保存
# 记录模型的特征重要性
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
# 获取特征重要性值
importances = perm_importance.importances_mean

# 对特征重要性进行排序
sorted_idx = np.argsort(importances)[::-1]

print(f'{model_name}_feature_names(top15):')
for i in feature_names[sorted_idx[:15]]:
    print(i)
print(f'average_{model_name}_feature_values(top15):')
for i in importances[sorted_idx[:15]]:
    print(i)

#记录更优的分数
best_r2_score=current_r2_score
best_mae_score=np.mean(mae_scores)
best_mape_score=np.mean(mape_scores)
if save_model:
    joblib.dump(model,   f'models/{model_name}/{model_name}_optimization1.pkl')  #保存模型
    # print(f'保存更优{model_name}模型文件') 
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均MAE分数:',best_mae_score)
"""
打印并输出分数至表格
"""


df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index = True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


文件夹models/knn已存在
The file 'scores_with_best_parameters.xlsx' exists.


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,frac f valence electrons,Grain Size,Precipitate Distribution,Habit Plane,Calculated Solid Solution,Calculated Grain Boundary,Calculated Yield Strength,屈服强度,抗拉强度 (UTS),追踪编号
0,12,30,18,12.131276,0.247009,12,52,73,21,68.266826,...,0.000000,4.00,0,0,17.669989,194.321983,226.991971,364.0,393.0,No630-6Al-1Zn-0.15Mn -3
1,12,64,52,12.922815,1.811506,12,23,68,45,67.259700,...,0.054984,36.00,3,1,57.602249,48.504397,121.106646,125.0,200.0,No232-Mg-9Gd-1Sm-0.5Zr-2
2,12,64,52,13.780587,3.401587,12,12,69,57,66.376359,...,0.078055,50.00,3,5,94.613373,69.789053,179.402426,200.0,340.0,No16-Mg–14Gd–3Y–1.8Zn–0.5Zr -9
3,12,64,52,13.179217,2.292508,12,12,68,56,66.814372,...,0.052520,1.50,2,1,76.138523,262.520047,353.658570,290.0,280.0,No96-Mg-9.3Gd-2.7Y-0.65Ag-0.52Zr-0.18Ce -4
4,12,30,18,12.112392,0.217496,12,52,73,21,68.118900,...,0.000000,15.00,0,0,11.604694,86.484728,113.089422,148.0,280.0,No460-3Al-1Zn-0.3Mn -22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,12,38,26,12.053936,0.104733,12,8,73,65,68.099036,...,0.000000,130.00,1,1,12.020723,23.509472,50.530195,58.5,220.0,No744-Mg-3Al-0.4Mn-0.05Sr
735,12,64,52,12.937303,1.825661,12,12,69,57,67.051202,...,0.033095,41.77,3,5,61.765688,53.715298,130.480986,223.0,295.0,No226-Mg-6Gd-3Y-1Zn-0.4Zr-0.7Ag-3
736,12,30,18,12.152650,0.281635,12,7,73,66,67.973850,...,0.000000,30.00,3,2,27.156102,71.029741,113.185842,91.1,144.3,No312-Mg-7.6Al-0.5Zn-1Ca-2
737,12,64,52,12.126080,0.237903,12,12,73,61,68.188783,...,0.003189,52.10,1,1,23.809563,45.036179,83.845741,125.0,217.0,No383-Mg-6Al-Zn-0.3Y-0.6Gd


{'algorithm': 'ball_tree', 'leaf_size': 30, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'n_neighbors': 8, 'p': 2, 'weights': 'distance'}
knn_feature_names(top15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
MagpieData range AtomicWeight
MagpieData maximum AtomicWeight
MagpieData range CovalentRadius
MagpieData maximum SpaceGroupNumber
MagpieData maximum CovalentRadius
MagpieData range SpaceGroupNumber
Habit Plane
MagpieData minimum MendeleevNumber
MagpieData avg_dev MeltingT
MagpieData range MendeleevNumber
MagpieData mean MeltingT
average_knn_feature_values(top15):
0.40065923036194406
0.22985945459061782
0.20760344500613703
0.06077643425921748
0.04935958518272727
0.049187559156948495
0.04663681441486496
0.03880876542790919
0.02874093893069911
0.028095208829404045
0.021692172601349806
0.013397767685456075
0.012262332522566833
0.012204321768755707
0.010004854091694175
平均 R2 分数: 0.537182749808483
平均 MAPE 分数: 16.13395051494306

In [51]:
df = pd.read_excel(score_path)
# 添加新数据的示例
new_data = {'Model': f'{model_name}', 'R^2': best_r2_score, 'MAPE': best_mape_score * 100, 'MAE': best_mae_score}
new_data_df = pd.DataFrame([new_data])

# 使用 pd.concat() 合并原 DataFrame 和新的 DataFrame
df = pd.concat([df, new_data_df], ignore_index=True)
# 保存数据框到Excel表格
df.to_excel(score_path, index = False)


In [58]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_validate, KFold
from sklearn.inspection import permutation_importance
import numpy as np

# 定义参数
params = {'algorithm': 'ball_tree', 'leaf_size': 30, 'metric': 'minkowski', 'metric_params': None, 'n_jobs': None, 'p': 2, 'weights': 'distance'}

# 定义交叉验证
kf = KFold(n_splits=5, shuffle=True, random_state=200)

# 定义评分指标
scoring = ['r2', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error']
df = pd.read_excel('train_set_new.xlsx', index_col=0)  # 引入这一列之后，原本的第一列就是一个索引，第0列会从有意义的列开始

# 定义特征和目标变量
X = df.drop(columns=['Precipitate Distribution',
      '屈服强度', '抗拉强度 (UTS)', '追踪编号'])  # 特征: 最后四列之前的所有列
y = df.iloc[:, -2]  # 目标: 倒数第四列
# 初始化记录结果的变量
best_r2_score = float('-inf')
best_n_neighbors = None
r2_scores_ls = []
mape_scores = []
mae_scores = []

# 循环不同的n_neighbors值
for n_neighbors in range(3, 13):
    model = KNeighborsRegressor(n_neighbors=n_neighbors, **params)
    model.fit(X_train, y_train)

    # 计算分折交叉验证结果
    results = cross_validate(model, X, y, cv=kf, scoring=scoring, return_train_score=False)

    # 输出分折交叉验证结果
    for i, (r2, mae, mape) in enumerate(zip(results['test_r2'], results['test_neg_mean_absolute_error'], results['test_neg_mean_absolute_percentage_error']), start=1):
        # 记录不同折的分数结果
        r2_scores_ls.append(r2)
        mape_scores.append(-mape)
        mae_scores.append(-mae)
    
    current_r2_score = np.mean(r2_scores_ls)
    
    if current_r2_score > best_r2_score:
        best_r2_score = current_r2_score
        best_mae_score = np.mean(mae_scores)
        best_mape_score = np.mean(mape_scores)
        best_n_neighbors = n_neighbors

        # 记录模型的特征重要性
        perm_importance = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
        importances = perm_importance.importances_mean
        sorted_idx = np.argsort(importances)[::-1]

        print(f'KNeighborsRegressor with n_neighbors={n_neighbors} feature importance (top 15):')
        for i in feature_names[sorted_idx[:15]]:
            print(i)
        print(f'Average feature importances (top 15):')
        for i in importances[sorted_idx[:15]]:
            print(i)

print('最佳 n_neighbors:', best_n_neighbors)
print('平均 R2 分数:', best_r2_score)
print('平均 MAPE 分数:', best_mape_score * 100)  # 转换为百分比
print('平均 MAE 分数:', best_mae_score)

KNeighborsRegressor with n_neighbors=3 feature importance (top 15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
MagpieData range CovalentRadius
Habit Plane
MagpieData range AtomicWeight
MagpieData maximum AtomicWeight
MagpieData range SpaceGroupNumber
MagpieData maximum CovalentRadius
MagpieData maximum SpaceGroupNumber
Lambda entropy
MagpieData minimum Electronegativity
MagpieData minimum Number
Shear modulus delta
Average feature importances (top 15):
0.529564283597412
0.3117609957659261
0.2513730098280663
0.08540707062247689
0.04521755284811301
0.0450458160773918
0.03037853646480189
0.03029740671790672
0.02486774346531664
0.023443630305523808
0.009416093782073287
0.0028094010978257883
0.0005243995244314921
0.00021745245754678882
0.00021225357343910659
KNeighborsRegressor with n_neighbors=4 feature importance (top 15):
Calculated Yield Strength
Calculated Grain Boundary
Grain Size
Calculated Solid Solution
MagpieData range CovalentRadius
H

In [59]:
joblib.dump(model, f'models/knn/knn_optimization_best.pkl')  #保存模型

['models/knn/knn_optimization_best.pkl']